# **Modeling a basic problem with FICO&reg; Xpress Optimizer - Python Interface**

***modeling.ipynb***

This example demonstrates how variables, or arrays thereof, and constraints, or arrays of constraints, are added to a problem. It prints the solution and attributes of the problem.

&copy; Copyright 2025-2026 Fair Isaac Corporation. The use of this example is subject to [legal and license requirements](https://github.com/fico-xpress/python-notebooks#legal-and-license-requirements).

In [ ]:
# Install the xpress package
%pip install -q xpress

Start by importing the xpress Python package with the alias *xp*, declare data and create an Xpress problem instance using [xpress.problem()](https://www.fico.com/fico-xpress-optimization/docs/latest/solver/optimizer/python/HTML/xpress.problem.html). A name can be assigned to a problem upon creation using the <tt>name</tt> argument:

In [ ]:
import xpress as xp

N = 4
S = range(N)

p = xp.problem(name="My first problem")

Add both v, an array (list) of variables, and v1 and v2, two scalar variables using the method [p.addVariable()](https://www.fico.com/fico-xpress-optimization/docs/latest/solver/optimizer/python/HTML/problem.addVariable.html).

In [ ]:
v = [p.addVariable(name="y{0}".format(i), lb=0, ub=2*N) for i in S]

v1 = p.addVariable(name="v1", lb=0, ub=10, vartype=xp.continuous)
v2 = p.addVariable(name="v2", lb=1, ub=7, threshold=3, vartype=xp.semicontinuous)
vb = p.addVariable(name="vb", vartype=xp.binary)

Add a list of constraints, which are given as arguments to [p.addConstraint()](https://www.fico.com/fico-xpress-optimization/docs/latest/solver/optimizer/python/HTML/problem.addConstraint.html): 
  - one explicitly created and stored in <tt>c1</tt>
  - two constraints passed directly as arguments: 
    - one using the scalar variables <tt>v1</tt> and <tt>v2</tt>
    - another one using selected elements of set <tt>v</tt>
  - a set (list) of constraints indexed by all ${i \in S: i<N-1}$ (recall that ranges in Python are numbered from 0 to N-1) 

In [ ]:
c1 = v1 + v2 >= 5

p.addConstraint(c1,
                2*v1 + 3*v2 >= 5,
                v[0] + v[2] >= 1,
                [v[i+1] >= v[i] + 1 for i in S if i < N-1])

(R1, R2, R3, [R4, R5, R6])

The method [problem.setObjective()]() sets the objective function of the problem.

In [ ]:
p.setObjective(xp.Sum([i*v[i] for i in S]), sense=xp.minimize)

Optimize and print the solve and solution statuses using the objects returned by [problem.optimize()](https://www.fico.com/fico-xpress-optimization/docs/latest/solver/optimizer/python/HTML/problem.optimize.html) or by querying problem attributes. The method [problem.getSolution()](https://www.fico.com/fico-xpress-optimization/docs/latest/solver/optimizer/python/HTML/problem.getSolution.html) returns the optimal solution as a list.

In [ ]:
solvestatus, solstatus = p.optimize()

if solvestatus == xp.SolveStatus.COMPLETED:
    print("Solve completed with solution status: ", solstatus.name)
else:
    print("Solve status: ", solvestatus.name)

# or alternatively
# print("Solve status: ", p.attributes.solvestatus.name)
# print("Solution status: ", p.attributes.solstatus.name)

print("Solution:", p.getSolution())

FICO Xpress v9.9.x, Hyper, solve started 15:17:55, Apr 9, 2026
Heap usage: 431KB (peak 431KB, 92KB system)
Minimizing MILP My first problem using up to 20 threads and up to 31GB memory, with these control settings:
OUTPUTLOG = 1
NLPPOSTSOLVE = 1
XSLP_DELETIONCONTROL = 0
XSLP_OBJSENSE = 1
Original problem has:
         6 rows            7 cols           12 elements         2 entities
Presolved problem has:
         0 rows            0 cols            0 elements         0 entities
Presolve finished in 0 seconds
Heap usage: 1525KB (peak 1540KB, 92KB system)
Will try to keep branch and bound tree memory usage below 24.0GB
Starting concurrent solve with dual (1 thread)

 Concurrent-Solve,   0s
            Dual        
    objective   dual inf
                        
------- optimal --------
Concurrent statistics:
           Dual: 0 simplex iterations, 0.00s
Optimal solution found
 
   Its         Obj Value      S   Ninf  Nneg   Sum Dual Inf  Time
     0         14.000000      D      0     

## Naming constraints

Starting from Xpress version 9.9, the `name` argument of [`addConstraint()`](https://www.fico.com/fico-xpress-optimization/docs/latest/solver/optimizer/python/HTML/problem.addConstraint.html) lets you attach a human-readable label to each constraint at creation time.

There are three forms:

| Form | Code | Names generated |
|------|------|-----------------|
| Single constraint | `addConstraint(expr, name="budget")` | `budget` |
| Collection with a list of names | `addConstraint(exprs, name=["name_0", "name_1", ...])` | `name_0`, `name_1`, ... |
| Collection with a prefix | `addConstraint((expr for i in S), name="cap")` | `cap(0)`, `cap(1)`, ... |

Named constraints appear, for example, in LP/MPS files or IIS diagnostic output. Names can be retrieved via the constraint object's `.name` attribute.

In [ ]:
q = xp.problem()
q.controls.outputlog = 0

x = q.addVariable(name="x", lb=0)
y = q.addVariable(name="y", lb=0)

# --- Form 1: single constraint with a plain name ---
budget = q.addConstraint(x + y <= 100, name="budget")
print(f"name: {budget.name!r:20s}  rhs: {budget.rhs}")

# --- Form 2: collection with a list of names ---
PERIODS = range(4)
demand  = [20, 35, 15, 40]
qty = q.addVariables(len(PERIODS), lb=0, name="qty")
q.addConstraint(qty >= demand, name=[f"min_demand_{t}" for t in PERIODS])

# --- Form 3: collection with a prefix --- solver appends (0), (1), ... ---
q.addConstraint((qty[t] <= 50 for t in PERIODS), name="capacity")

q.setObjective(xp.Sum(qty[t] for t in PERIODS), sense=xp.minimize)
q.optimize()

# --- Inspect all constraint names via constraint objects ---
print("All constraint names:")
for i, c in enumerate(q.getConstraint(list(range(q.attributes.rows)))):
    print(f"  row {i}: {c.name!r}")

name: 'budget'              rhs: 100.0
All constraint names:
  row 0: 'budget'
  row 1: 'min_demand_0'
  row 2: 'min_demand_1'
  row 3: 'min_demand_2'
  row 4: 'min_demand_3'
  row 5: 'capacity(0)'
  row 6: 'capacity(1)'
  row 7: 'capacity(2)'
  row 8: 'capacity(3)'
